# 03 · Test predictive value

We test whether three features derived from past SPY prices help classify a future price movement. Seven fixed classifiers are compared with a simple training-majority baseline using chronological training and validation periods.


## 1. Data and setup

Use the supplied daily historical SPY closing-price series, bounded at **2021-12-28**. The setup cell establishes repository paths and the output directory; the data cell loads prices within this cutoff.

Running this notebook requires an authorized copy of `data/raw/tpq_eod_2014_2023.csv`. See [the teaching setup instructions](README.md) for the expected SHA-256.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src/quant_journey').is_dir():
    ROOT = next(
        parent for parent in ROOT.parents
        if (parent / 'src/quant_journey').is_dir()
    )
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/raw/tpq_eod_2014_2023.csv'
OUT = ROOT / 'outputs/track_e_multifamily'
OUT.mkdir(parents=True, exist_ok=True)
CUTOFF = '2021-12-28'


In [ ]:
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from src.quant_journey.data.tpq import load_spy_prices_through
from src.quant_journey.execution.dataset import make_execution_dataset
from src.quant_journey.validation.label_aware import split_train_validation_by_dates

prices = load_spy_prices_through(DATA, CUTOFF)


**Interpretation.** This bounded daily price series is the input for both the past-price features and the later hypothetical target. All records used below stay within the stated cutoff.


## 2. Define the prediction question

At the end of session *t*, use only information then available to classify whether SPY rises from close $t+1$ to close $t+2$:

$$
y_t = \mathbf{1}\left[\frac{P_{t+2}}{P_{t+1}} - 1 > 0\right]
$$

Define $y_t=1$ as a positive movement and $y_t=0$ otherwise. The target is observed from close *t+1* to close *t+2*. This matches Notebook 04’s hypothetical timing: enter at the next close, then value the position at the following close. It is a close-only price-change proxy, not a claim about an achievable fill.


## 3. Construct the three predictors

All three inputs use SPY closes available by the end of session $t$. Let $P_t$ denote the close and define the one-session percentage change as $r_t = \frac{P_t}{P_{t-1}} - 1$:

- `return_1d`: $r_t$, the latest one-session percentage change.
- `momentum_5d`: $\frac{P_t}{P_{t-5}} - 1$, the five-session close-to-close percentage change.
- `volatility_5d`: the sample standard deviation (Pandas `ddof=1`) of $r_{t-4},\ldots,r_t$, the latest five one-session percentage changes.

Notebook 02’s SMA, RSI, MACD, PCA and clusters are exploratory descriptions; none is an input to this fixed comparison.


In [ ]:
data = make_execution_dataset(prices)


## 4. Separate training and validation

Use feature dates before **2020-01-02** for training and dates from **2020-01-02** to before **2021-12-29** for validation. Keep a training label only when its `label_available_at` date is strictly before validation starts, removing training labels that mature into validation. Validation labels must mature before the reserved boundary. Training contains 1,503 rows (**2014-01-09–2019-12-27**); validation contains 500 rows (**2020-01-02–2021-12-23**). Any standardizer is inside a model pipeline and is fitted on training rows only. Validation has previously been examined; the reserved test remains unused.


In [ ]:
train, validation = split_train_validation_by_dates(
    data, '2020-01-02', '2021-12-29'
)
features = ['return_1d', 'momentum_5d', 'volatility_5d']
target = 'execution_target_up'
X_train = train[features]
y_train = train[target]
X_val = validation[features]
y_val = validation[target]
print(
    'train/validation:', X_train.shape, X_val.shape,
    'date ranges:', train.index.min().date(), train.index.max().date(),
    validation.index.min().date(), validation.index.max().date(),
)


**Interpretation.** Feature dates determine the initial partitions. The purge then retains training rows with `label_available_at < 2020-01-02` and validation rows with `label_available_at < 2021-12-29`. A label that matures on the boundary is excluded. Validation is prior development evidence, not a fresh holdout.


## 5. Define the seven classifiers and baseline

The fixed families illustrate different classification approaches: GaussianNB models class-conditional feature distributions; logistic regression uses a linear log-odds boundary; a decision tree makes threshold-based splits; a random forest combines trees; LinearSVC separates classes with a linear margin; hard voting combines GaussianNB, logistic regression, random forest and LinearSVC; and the MLP is a feed-forward neural network. The table summarizes the fixed canonical configuration. Logistic regression, LinearSVC and the MLP each use a train-fitted standardizer. Logistic regression, the decision tree, the forest, LinearSVC and the MLP use seed 42. The hard vote uses the same preprocessing and settings for its four members. Both the forest and the voting estimator use one job.

The baseline always predicts the most common training class. This is a fixed educational comparison: no tuning, winner selection or validation-based setting changes are performed.


In [ ]:
# Fixed canonical estimator and preprocessing settings.
nb = GaussianNB()
logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', random_state=42),
)
tree = DecisionTreeClassifier(random_state=42)
forest = RandomForestClassifier(
    n_estimators=30, max_depth=9, random_state=42, n_jobs=1
)
svc = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0, dual=False, max_iter=10000, class_weight='balanced',
        random_state=42,
    ),
)
voting = VotingClassifier(
    estimators=[
        ('nb', GaussianNB()),
        ('logit', make_pipeline(
            StandardScaler(),
            LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', random_state=42),
        )),
        ('rf', RandomForestClassifier(
            n_estimators=30, max_depth=9, random_state=42, n_jobs=1,
        )),
        ('svc', make_pipeline(
            StandardScaler(),
            LinearSVC(
                C=1.0, dual=False, max_iter=10000, class_weight='balanced',
                random_state=42,
            ),
        )),
    ],
    voting='hard',
    n_jobs=1,
)
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(32, 24, 16), activation='relu', solver='adam',
        learning_rate_init=.001, max_iter=500, random_state=42,
    ),
)
models = {
    'GaussianNB': nb,
    'Logistic regression': logit,
    'Decision tree': tree,
    'Random forest': forest,
    'LinearSVC': svc,
    'Hard voting': voting,
    'MLP': mlp,
}
settings = pd.DataFrame([
    ['GaussianNB', 'sklearn defaults'],
    ['Logistic regression', 'C=1; lbfgs; max_iter=500; scaled'],
    ['Decision tree', 'sklearn defaults; seed=42'],
    ['Random forest', '30 trees; max_depth=9; seed=42'],
    ['LinearSVC', 'C=1; dual=False; max_iter=10000; balanced; scaled'],
    ['Hard voting', 'hard vote: NB, logistic, RF, LinearSVC'],
    ['MLP', '32-24-16; relu; adam; lr=.001; max_iter=500; scaled; seed=42'],
], columns=['family', 'fixed settings'])
display(settings)


**Interpretation.** The MLP allows up to 500 epochs. In the training-only convergence diagnostic, it met the stopping criterion at epoch 364 on both Apple Accelerate and OpenBLAS. The training-majority baseline is a reference prediction rule, not an eighth fitted model.


## 6. Fit, predict and compare

Fit each family once on the shared training rows and predict the same validation dates. **Accuracy** is the fraction of validation labels classified correctly. **Balanced accuracy** averages recall across the two classes, giving each class equal weight. Each confusion matrix shows actual classes by rows and predicted classes by columns; all seven classifiers and the training-majority baseline are shown.


In [ ]:
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    # Each fitted classifier predicts on the same validation rows.
    predictions[name] = pd.Series(
        model.predict(X_val).astype(int), index=X_val.index, name=name
    )
    print(name, 'fit complete')
predictions['Training majority baseline'] = pd.Series(
    int(y_train.mode().iloc[0]), index=X_val.index
)
metrics = []
for name, pred in predictions.items():
    metrics.append({
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, pred),
        'predicted_up': int(pred.sum()),
    })
metrics = pd.DataFrame(metrics).set_index('model')
display(metrics)
metrics.to_csv(OUT / 'validation_metrics.csv', index_label='model')


**Validation results (MLP cap of 500).** No fitted model exceeds the baseline’s **0.580 accuracy**. Logistic regression ties it at 0.580; the other accuracies are lower. Balanced accuracies range from about 0.490 to 0.515 around the baseline’s 0.500. These small differences do not establish reliable skill. Both logistic regression and the baseline predict UP for all 500 validation observations; each has confusion matrix

$$
\begin{bmatrix}
0 & 210 \\
0 & 290
\end{bmatrix}
$$

in label order [0, 1].

The MLP met its training stopping criterion at epoch 364. It predicted UP on 285 of 500 validation dates, with accuracy 0.502 and balanced accuracy 0.490558. Compared with the cap-200 run, 61 MLP predictions changed and all other prediction columns matched exactly.

The cap-500 run emitted 24 matrix-operation `RuntimeWarning`s during fitting and prediction, eight each for divide by zero, overflow and invalid value. The MLP emitted no convergence warning. The cause of the matrix warnings remains unresolved.

In the earlier cap-200 comparison, the project and OpenBLAS runs agreed on all eight prediction columns and their metrics. OpenBLAS emitted no matrix-operation warnings. The cap-500 validation predictions have been checked on the project backend only.

<details>
<summary>Prior run (MLP cap of 200): matrix-operation and MLP convergence warning output (25 occurrences)</summary>

<pre>/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:164: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:164: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:164: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:164: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:164: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:164: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn&#x27;t converged yet.
  warnings.warn(
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/frasagui/Projects/A-Quant-Journey/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:208: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b</pre>
</details>


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    matrix = confusion_matrix(y_val, pred, labels=[0, 1])
    ax.imshow(matrix, cmap='Blues')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('predicted')
    ax.set_ylabel('actual')
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center')
plt.tight_layout()


**Interpretation.** The panels keep each fitted family beside the simple reference. The all-UP logistic and baseline panels make their identical class predictions visible, including the 210 validation non-UP labels they miss. In the revised MLP panel, the actual non-UP row contains 88 predicted non-UP and 122 predicted UP. The actual UP row contains 127 predicted non-UP and 163 predicted UP.


## 7. Save predictions for portfolio accounting

We save each model’s prediction by feature date, alongside the observed label. Notebook 04 loads these predictions to construct hypothetical positions. Separate files record classification metrics and the experiment configuration. The manifest copies the expected snapshot digest into `data_sha256`. Verification against the input file bytes is reserved for the final release audit.


In [ ]:
# Persist predictions and provenance.
pred_path = OUT / 'validation_predictions.csv'
pd.DataFrame(predictions).assign(actual=y_val).to_csv(
    pred_path,
    index_label='feature_date',
)
manifest = {
    'validation_predictions_csv_sha256': hashlib.sha256(pred_path.read_bytes()).hexdigest(),
    'data_sha256': '771c40d3f8d19d1cf524764296bfa2943f8ca9c63514b99bb8d95c9b84d1e407',
    'cutoff': CUTOFF,
    'features': features,
    'target': 'close[t+2]/close[t+1]-1 > 0',
    'train_rows': len(train),
    'validation_rows': len(validation),
    'models': settings.to_dict(orient='records'),
    'reserved_test_access': False,
    'validation_is_fresh_holdout': False,
}
_ = (OUT / 'provenance.json').write_text(
    json.dumps(manifest, indent=2)
)


**Interpretation.** The existing artifacts remain `validation_predictions.csv` (feature-date index, model/baseline columns and `actual`), `validation_metrics.csv` (accuracy, balanced accuracy and predicted-UP count), and `provenance.json`. The manifest records the SHA-256 digest of the saved prediction CSV.


## 8. Findings and next step

On these previously examined validation dates with the MLP capped at 500 epochs, none of the seven fixed classifiers improves on the training-majority baseline’s accuracy, and the small balanced-accuracy differences do not establish reliable predictive skill. Notebook 04 takes the dated predictions into hypothetical positions and examines how timing and stated costs affect the resulting equity paths.
